# So sánh Random Forest, Logistic Regression và XGBoost trong dự đoán nguy cơ mắc bệnh tim

**Dữ liệu:** `heart.csv` (918 bệnh nhân, 11 đặc trưng, nhãn `HeartDisease`).

**Mục tiêu:** chạy ba thuật toán trên **cùng một cách chia dữ liệu, cùng các fold**, so sánh công bằng và **chọn ra mô hình tối ưu nhất** theo một quy tắc minh bạch.

## Quy trình

| Bước | Nội dung |
|---|---|
| 1 | Nạp dữ liệu, EDA, tiền xử lí |
| 2 | Chia **Train 70% / Validation 15% / Test 15%** (phân tầng) |
| 3 | Với mỗi mô hình: baseline + **GridSearchCV (Stratified 5-Fold) trên Train** |
| 4 | Đánh giá trên **Validation** |
| 5 | **Repeated Stratified K-Fold (5 x 5)** trên Train+Val cho cùng một bộ fold: ước lượng ổn định kèm độ lệch chuẩn |
| 6 | **Kiểm định thống kê ghép cặp** (corrected resampled t-test) giữa các mô hình |
| 7 | Huấn luyện lại trên Train+Val, đánh giá **một lần** trên Test: ROC chung, ma trận nhầm lẫn, khoảng tin cậy bootstrap, kiểm định McNemar |
| 8 | So sánh độ quan trọng đặc trưng giữa các mô hình |
| 9 | **Chọn mô hình tối ưu** theo quy tắc tự động, lưu mô hình và bảng kết quả |

## Quy tắc chọn "mô hình tối ưu"

Độ chính xác của các mô hình trên bộ dữ liệu nhỏ thường chênh nhau rất ít, nên không chọn theo con số cao nhất một cách máy móc. Quy tắc dùng ở đây:

1. Xếp hạng theo **ROC-AUC trung bình** trong Repeated K-Fold (`PRIMARY_METRIC`).
2. Giữ lại các mô hình **không kém hơn một cách có ý nghĩa thống kê** so với mô hình đứng đầu (p >= `ALPHA`).
3. Loại những mô hình có **Recall** dưới ngưỡng `MIN_RECALL` (không bỏ sót quá nhiều ca bệnh).
4. Nếu còn nhiều mô hình ngang nhau, chọn mô hình **đơn giản/dễ giải thích nhất** theo `PARSIMONY_ORDER` (nguyên tắc tiết kiệm: cùng hiệu năng thì chọn mô hình đơn giản hơn).

Các tham số này nằm ở ô cấu hình, bạn có thể đổi.

## 1. Cấu hình, nạp thư viện và dữ liệu

In [ ]:
# Colab thường đã cài sẵn xgboost; nếu chưa có thì tự cài
try:
    import xgboost
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost

import warnings, time, itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy import stats

from sklearn.base import clone
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV, cross_validate)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)

# ---------------- CẤU HÌNH ----------------
SEED = 42
N_SPLITS = 5
N_REPEATS = 5                  # số lần lặp Repeated K-Fold (5 -> 25 fold, ~5-8 phút; tăng lên 10 cho kết quả chắc hơn)
PRIMARY_METRIC = "roc_auc"     # tiêu chí chính: "roc_auc", "accuracy", "f1" hoặc "recall"
ALPHA = 0.05                   # mức ý nghĩa thống kê
MIN_RECALL = 0.85              # ngưỡng Recall tối thiểu (hạn chế bỏ sót ca bệnh)
PARSIMONY_ORDER = ["Logistic Regression", "Random Forest", "XGBoost"]   # từ đơn giản đến phức tạp
# ------------------------------------------

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("xgboost", xgboost.__version__)

In [ ]:
# Trên Google Colab: chọn file heart.csv từ máy tính khi hộp thoại hiện ra.
try:
    from google.colab import files
    import os
    if not os.path.exists("heart.csv"):
        uploaded = files.upload()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

df_raw = pd.read_csv("heart.csv")
print("Kích thước:", df_raw.shape)
df_raw.head()

## 2. Khám phá nhanh dữ liệu (EDA)

In [ ]:
print("Dòng trùng lặp:", df_raw.duplicated().sum())
print("Cholesterol = 0:", (df_raw["Cholesterol"] == 0).sum(), "| RestingBP = 0:", (df_raw["RestingBP"] == 0).sum())
print("Tỉ lệ bệnh khi Cholesterol = 0: %.1f%% | khi Cholesterol > 0: %.1f%%" % (
    100*df_raw.loc[df_raw.Cholesterol == 0, "HeartDisease"].mean(),
    100*df_raw.loc[df_raw.Cholesterol > 0, "HeartDisease"].mean()))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df_raw["HeartDisease"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#4C9F70", "#D1495B"])
axes[0].set_title("Phân bố nhãn"); axes[0].set_xticklabels(["0: không bệnh", "1: có bệnh"], rotation=0)
sns.countplot(data=df_raw, x="ST_Slope", hue="HeartDisease", ax=axes[1], palette=["#4C9F70", "#D1495B"])
axes[1].set_title("ST_Slope theo nhãn")
sns.countplot(data=df_raw, x="ChestPainType", hue="HeartDisease", ax=axes[2], palette=["#4C9F70", "#D1495B"])
axes[2].set_title("ChestPainType theo nhãn")
plt.tight_layout(); plt.show()

## 3. Tiền xử lí và chia dữ liệu

| Vấn đề | Cách xử lí |
|---|---|
| `Cholesterol = 0`, `RestingBP = 0` (thiếu ngầm) | Chuyển thành `NaN`, điền **median** (học từ tập train, nằm trong `Pipeline`) |
| Việc *thiếu* Cholesterol mang thông tin | Thêm cột cờ `Cholesterol_missing` |
| Biến phân loại | **One-Hot Encoding** |
| Biến số | Chuẩn hoá (`StandardScaler`) **chỉ cho Logistic Regression**; Random Forest và XGBoost dựa trên cây nên không cần |
| Dòng trùng lặp | Loại bỏ |

Điền thiếu, One-Hot và chuẩn hoá đều nằm trong `Pipeline` nên mỗi fold chỉ học từ dữ liệu train của chính nó (không rò rỉ dữ liệu).

In [ ]:
def clean(data: pd.DataFrame) -> pd.DataFrame:
    """Làm sạch theo từng dòng (không học gì từ dữ liệu => không rò rỉ)."""
    d = data.copy()
    d["Cholesterol_missing"] = (d["Cholesterol"] == 0).astype(int)
    d["Cholesterol"] = d["Cholesterol"].replace(0, np.nan)
    d["RestingBP"] = d["RestingBP"].replace(0, np.nan)
    return d

df = clean(df_raw.drop_duplicates().reset_index(drop=True))
X, y = df.drop(columns="HeartDisease"), df["HeartDisease"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.15/0.85, stratify=y_trainval, random_state=SEED)

pd.DataFrame({
    "Số mẫu": [len(X_train), len(X_val), len(X_test)],
    "Tỉ lệ (%)": [round(100*len(s)/len(X), 1) for s in (X_train, X_val, X_test)],
    "Tỉ lệ bệnh (%)": [round(100*s.mean(), 1) for s in (y_train, y_val, y_test)],
}, index=["Train", "Validation", "Test"])

## 4. Định nghĩa ba mô hình và lưới siêu tham số

In [ ]:
num_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
pass_cols = ["FastingBS", "Cholesterol_missing"]          # đã là 0/1

def make_prep(scale: bool):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("pass", "passthrough", pass_cols),
    ])

models = {
    "Random Forest": Pipeline([("prep", make_prep(False)),
                               ("clf", RandomForestClassifier(random_state=SEED, n_jobs=1))]),
    "Logistic Regression": Pipeline([("prep", make_prep(True)),
                                     ("clf", LogisticRegression(max_iter=5000, random_state=SEED))]),
    "XGBoost": Pipeline([("prep", make_prep(False)),
                         ("clf", XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=1))]),
}

grids = {
    "Random Forest": {
        "clf__n_estimators": [200, 500],
        "clf__max_depth": [None, 5, 8],
        "clf__min_samples_leaf": [1, 2, 4],
        "clf__max_features": ["sqrt", "log2"],
    },
    "Logistic Regression": {
        "clf__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100],
        "clf__class_weight": [None, "balanced"],
    },
    "XGBoost": {     # lưới mở rộng về phía "đơn giản" vì cấu hình tốt nhất thường nằm sát biên
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [1, 2, 3],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
    },
}
print({k: int(np.prod([len(v) for v in g.values()])) for k, g in grids.items()}, "tổ hợp tham số")

## 5. Baseline và tinh chỉnh bằng Stratified K-Fold trên tập Train

Mỗi mô hình chạy GridSearchCV với cùng `StratifiedKFold(5)`. Tiêu chí chọn tham số (`refit`) là `PRIMARY_METRIC`.

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc"}

baseline_cv, tuned, best_params, tune_info = {}, {}, {}, {}
for name, pipe in models.items():
    t0 = time.time()
    cvb = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    baseline_cv[name] = {m: cvb["test_" + m].mean() for m in scoring}

    g = GridSearchCV(pipe, grids[name], cv=skf, scoring=scoring, refit=PRIMARY_METRIC, n_jobs=-1)
    g.fit(X_train, y_train)
    tuned[name], best_params[name] = g.best_estimator_, g.best_params_
    tune_info[name] = {"CV baseline (Train)": baseline_cv[name][PRIMARY_METRIC],
                       "CV tinh chỉnh (Train)": g.best_score_,
                       "Thời gian (s)": round(time.time() - t0, 1)}
    print(f"{name}: {g.best_params_}")

pd.DataFrame(tune_info).T.round(4).rename_axis(f"{PRIMARY_METRIC} (K-Fold trên Train)")

In [ ]:
# Cảnh báo nếu tham số tốt nhất nằm sát biên lưới (lưới có thể chưa đủ rộng)
for name, bp in best_params.items():
    edge = [k.replace("clf__", "") for k, v in bp.items()
            if isinstance(v, (int, float)) and not isinstance(v, bool)
            and len(grids[name][k]) > 2 and v in (min(x for x in grids[name][k] if x is not None),
                                                     max(x for x in grids[name][k] if x is not None))]
    if edge:
        print(f"[{name}] tham số nằm sát biên lưới: {edge}")

## 6. Đánh giá trên tập Validation

In [ ]:
def evaluate(y_true, y_pred, proba):
    return {"Accuracy":  accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall":    recall_score(y_true, y_pred),
            "F1":        f1_score(y_true, y_pred),
            "ROC-AUC":   roc_auc_score(y_true, proba)}

val_rows = {n: evaluate(y_val, m.predict(X_val), m.predict_proba(X_val)[:, 1]) for n, m in tuned.items()}
val_table = pd.DataFrame(val_rows).T
display(val_table.round(4))

fig, ax = plt.subplots(figsize=(5.5, 5))
for n, m in tuned.items():
    RocCurveDisplay.from_predictions(y_val, m.predict_proba(X_val)[:, 1],
                                     name=f"{n} (AUC={val_table.loc[n, 'ROC-AUC']:.3f})", ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_title("Đường ROC - Validation")
plt.tight_layout(); plt.show()

## 7. Repeated Stratified K-Fold (5 x `N_REPEATS`) trên Train + Validation

Đây là phần so sánh **đáng tin nhất**: mỗi mô hình (với tham số đã chọn) được đánh giá trên **25 fold** (mặc định), và cả ba mô hình dùng **đúng cùng các fold** nên các kết quả ghép cặp được với nhau.

> Lưu ý: tham số được chọn trên tập Train, rồi mới đánh giá trên Train+Val nên ước lượng có thể lạc quan nhẹ. Mức lạc quan này như nhau cho cả ba mô hình nên không làm lệch phép so sánh.

In [ ]:
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)

rcv = {}
for name in models:
    est = clone(models[name]).set_params(**best_params[name])
    rcv[name] = cross_validate(est, X_trainval, y_trainval, cv=rskf, scoring=scoring, n_jobs=-1)
    print(f"{name}: xong {len(rcv[name]['test_accuracy'])} fold")

disp = {"accuracy": "Accuracy", "precision": "Precision", "recall": "Recall", "f1": "F1", "roc_auc": "ROC-AUC"}
cv_mean = pd.DataFrame({n: {disp[m]: rcv[n]["test_" + m].mean() for m in scoring} for n in models}).T
cv_std = pd.DataFrame({n: {disp[m]: rcv[n]["test_" + m].std() for m in scoring} for n in models}).T

print(f"Trung bình ± độ lệch chuẩn qua {N_SPLITS * N_REPEATS} fold:")
display((cv_mean.round(4).astype(str) + " ± " + cv_std.round(4).astype(str)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

cv_mean.plot(kind="bar", yerr=cv_std, ax=axes[0], capsize=3, rot=0)
axes[0].set_ylim(0.7, 1.0); axes[0].set_title("Chỉ số trung bình ± độ lệch chuẩn (Repeated K-Fold)")
axes[0].legend(loc="lower right", ncol=3, fontsize=8)

long = pd.DataFrame([{"Mô hình": n, "ROC-AUC": v} for n in models for v in rcv[n]["test_roc_auc"]])
sns.boxplot(data=long, x="Mô hình", y="ROC-AUC", ax=axes[1], palette="Set2")
sns.stripplot(data=long, x="Mô hình", y="ROC-AUC", ax=axes[1], color="k", alpha=0.25, size=3)
axes[1].set_title("Phân bố ROC-AUC trên từng fold")
plt.tight_layout(); plt.show()

## 8. Kiểm định thống kê ghép cặp giữa các mô hình

Dùng **corrected resampled t-test** (Nadeau & Bengio, 2003). Các fold của cross-validation dùng chung dữ liệu nên không độc lập; t-test thường sẽ cho p-value quá nhỏ. Phiên bản hiệu chỉnh thêm số hạng `n_test / n_train` vào phương sai để tránh kết luận "khác biệt có ý nghĩa" một cách sai lầm.

- **H0:** hai mô hình có hiệu năng trung bình như nhau.
- p < `ALPHA`: có bằng chứng thống kê rằng một mô hình tốt hơn.
- p >= `ALPHA`: **không đủ bằng chứng** để phân biệt (không có nghĩa là chắc chắn bằng nhau).

In [ ]:
n_total = len(X_trainval)
n_test_fold = n_total / N_SPLITS
n_train_fold = n_total - n_test_fold

def corrected_ttest(d):
    """d: mảng chênh lệch ghép cặp (mô hình A trừ mô hình B) trên từng fold."""
    k, var = len(d), np.var(d, ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = np.mean(d) / np.sqrt((1 / k + n_test_fold / n_train_fold) * var)
    return t, 2 * stats.t.sf(abs(t), df=k - 1)

def pairwise_table(metric):
    names = list(models)
    out = pd.DataFrame("-", index=names, columns=names)
    for a, b in itertools.permutations(names, 2):
        d = rcv[a]["test_" + metric] - rcv[b]["test_" + metric]
        _, p = corrected_ttest(d)
        out.loc[a, b] = f"{d.mean():+.4f} (p={p:.3f})"
    return out

for metric in ["roc_auc", "accuracy", "recall"]:
    print(f"\n{disp[metric]}: chênh lệch trung bình (hàng trừ cột) và p-value")
    display(pairwise_table(metric))

## 9. Huấn luyện lại trên Train + Validation và đánh giá trên tập Test

Test được giữ kín từ đầu, chỉ dùng **một lần** ở bước này.

In [ ]:
final, fit_time, test_pred, test_proba = {}, {}, {}, {}
for name in models:
    est = clone(models[name]).set_params(**best_params[name])
    t0 = time.time()
    est.fit(X_trainval, y_trainval)
    fit_time[name] = time.time() - t0
    final[name] = est
    test_pred[name] = est.predict(X_test)
    test_proba[name] = est.predict_proba(X_test)[:, 1]

test_table = pd.DataFrame({n: evaluate(y_test, test_pred[n], test_proba[n]) for n in models}).T
display(test_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.6))
for n in models:
    RocCurveDisplay.from_predictions(y_test, test_proba[n],
                                     name=f"{n} (AUC={test_table.loc[n, 'ROC-AUC']:.3f})", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_title("Đường ROC chung - Test")
for ax, n in zip(axes[1:], models):
    ConfusionMatrixDisplay.from_predictions(y_test, test_pred[n], display_labels=["Không bệnh", "Có bệnh"],
                                            cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(n)
plt.tight_layout(); plt.show()

### 9.1. Khoảng tin cậy bootstrap 95% trên tập Test

Tập Test chỉ có vài trăm mẫu nên con số một lần đo có độ bất định lớn. Bootstrap (lấy mẫu lại 2000 lần, có hoàn lại) cho biết khoảng dao động hợp lí của từng chỉ số.

In [ ]:
rng = np.random.default_rng(SEED)
y_arr, n_t, B = y_test.values, len(y_test), 2000
boot = {n: {"Accuracy": [], "Recall": [], "ROC-AUC": []} for n in models}

for _ in range(B):
    idx = rng.integers(0, n_t, n_t)
    yt = y_arr[idx]
    if yt.min() == yt.max():
        continue
    for n in models:
        p, pr = test_pred[n][idx], test_proba[n][idx]
        boot[n]["Accuracy"].append((p == yt).mean())
        boot[n]["Recall"].append(((p == 1) & (yt == 1)).sum() / (yt == 1).sum())
        boot[n]["ROC-AUC"].append(roc_auc_score(yt, pr))

ci = pd.DataFrame({
    n: {k: f"{test_table.loc[n, k]:.3f}  [{np.percentile(v, 2.5):.3f} - {np.percentile(v, 97.5):.3f}]"
        for k, v in boot[n].items()} for n in models}).T
display(ci)

### 9.2. Kiểm định McNemar trên tập Test

So sánh từng cặp mô hình dựa trên các ca mà **một mô hình đúng, mô hình kia sai**. p < `ALPHA` nghĩa là hai mô hình mắc lỗi khác nhau một cách có ý nghĩa.

In [ ]:
rows = []
for a, b in itertools.combinations(models, 2):
    ok_a, ok_b = test_pred[a] == y_arr, test_pred[b] == y_arr
    only_a, only_b = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    disc = only_a + only_b
    p = stats.binomtest(min(only_a, only_b), disc, 0.5).pvalue if disc > 0 else 1.0
    rows.append({"Mô hình 1": a, "Mô hình 2": b, "Chỉ mô hình 1 đúng": only_a, "Chỉ mô hình 2 đúng": only_b,
                 "Số ca bất đồng": disc, "p-value": round(p, 3)})
display(pd.DataFrame(rows))

all_wrong = int(((test_pred["Random Forest"] != y_arr) & (test_pred["Logistic Regression"] != y_arr)
                 & (test_pred["XGBoost"] != y_arr)).sum())
print(f"Số ca cả ba mô hình cùng đoán sai: {all_wrong}/{n_t} (các ca 'khó' mà thuật toán nào cũng sai)")

## 10. So sánh độ quan trọng đặc trưng giữa các mô hình (permutation importance, Test)

In [ ]:
imp = {}
for n in models:
    r = permutation_importance(final[n], X_test, y_test, n_repeats=20, random_state=SEED,
                               scoring="roc_auc", n_jobs=-1)
    s = pd.Series(r.importances_mean, index=X_test.columns).clip(lower=0)
    imp[n] = s / s.sum()                     # chuẩn hoá để các mô hình so sánh được
imp = pd.DataFrame(imp)
order = imp.mean(axis=1).sort_values().index
imp.loc[order].plot(kind="barh", figsize=(9, 6.5), width=0.8)
plt.title("Độ quan trọng đặc trưng (chuẩn hoá, tổng = 1)"); plt.xlabel("Tỉ trọng")
plt.tight_layout(); plt.show()

print("Top 3 đặc trưng của từng mô hình:")
for n in models:
    print(f"  {n}: {', '.join(imp[n].sort_values(ascending=False).index[:3])}")

## 11. Chọn mô hình tối ưu

Áp dụng quy tắc đã nêu ở đầu notebook, hoàn toàn tự động dựa trên kết quả Repeated K-Fold.

In [ ]:
metric_name = disp[PRIMARY_METRIC]
mean_primary = {n: rcv[n]["test_" + PRIMARY_METRIC].mean() for n in models}
best_stat = max(mean_primary, key=mean_primary.get)

# (2) mô hình nào không kém hơn mô hình đứng đầu một cách có ý nghĩa thống kê?
p_vs_best = {}
for n in models:
    if n == best_stat:
        p_vs_best[n] = 1.0
    else:
        _, p_vs_best[n] = corrected_ttest(rcv[best_stat]["test_" + PRIMARY_METRIC] - rcv[n]["test_" + PRIMARY_METRIC])
not_worse = [n for n in models if p_vs_best[n] >= ALPHA]

# (3) lọc theo Recall tối thiểu
mean_recall = {n: rcv[n]["test_recall"].mean() for n in models}
pool = [n for n in not_worse if mean_recall[n] >= MIN_RECALL] or [best_stat]

# (4) nếu còn nhiều mô hình ngang nhau, chọn mô hình đơn giản nhất
recommended = sorted(pool, key=PARSIMONY_ORDER.index)[0]

interpret = {"Logistic Regression": "Cao (hệ số, odds ratio)",
             "Random Forest": "Trung bình (độ quan trọng đặc trưng)",
             "XGBoost": "Trung bình (độ quan trọng đặc trưng)"}

scoreboard = pd.DataFrame({
    f"CV {metric_name}": {n: f"{mean_primary[n]:.4f} ± {rcv[n]['test_' + PRIMARY_METRIC].std():.4f}" for n in models},
    "CV Accuracy": {n: f"{cv_mean.loc[n, 'Accuracy']:.4f}" for n in models},
    "CV Recall": {n: f"{mean_recall[n]:.4f}" for n in models},
    f"p so với '{best_stat}'": {n: ("-" if n == best_stat else f"{p_vs_best[n]:.3f}") for n in models},
    "Val ROC-AUC": {n: f"{val_table.loc[n, 'ROC-AUC']:.4f}" for n in models},
    "Test Accuracy": {n: f"{test_table.loc[n, 'Accuracy']:.4f}" for n in models},
    "Test Recall": {n: f"{test_table.loc[n, 'Recall']:.4f}" for n in models},
    "Test ROC-AUC": {n: f"{test_table.loc[n, 'ROC-AUC']:.4f}" for n in models},
    "Thời gian huấn luyện (s)": {n: f"{fit_time[n]:.2f}" for n in models},
    "Khả năng giải thích": interpret,
})
display(scoreboard)

print("=" * 70)
print(f"1) Cao nhất theo {metric_name} (CV): {best_stat} ({mean_primary[best_stat]:.4f})")
if len(not_worse) == 1:
    print("2) Mô hình này TỐT HƠN CÓ Ý NGHĨA THỐNG KÊ so với cả hai mô hình còn lại.")
else:
    others = [n for n in not_worse if n != best_stat]
    print(f"2) Không kém {best_stat} một cách có ý nghĩa thống kê (p >= {ALPHA}): {', '.join(others)}")
print(f"3) Sau khi lọc Recall >= {MIN_RECALL}: {', '.join(pool)}")
if len(pool) > 1:
    print(f"4) Các mô hình này NGANG NHAU về thống kê, chọn mô hình đơn giản nhất theo PARSIMONY_ORDER.")
print("=" * 70)
print(f">>> MÔ HÌNH TỐI ƯU ĐƯỢC CHỌN: {recommended}")
print("=" * 70)

In [ ]:
# Điểm số theo từng tiêu chí để dễ trình bày (1 = tốt nhất)
rank = pd.DataFrame({
    f"CV {metric_name}": pd.Series(mean_primary).rank(ascending=False),
    "CV Accuracy": cv_mean["Accuracy"].rank(ascending=False),
    "CV Recall": pd.Series(mean_recall).rank(ascending=False),
    "Test ROC-AUC": test_table["ROC-AUC"].rank(ascending=False),
    "Test Accuracy": test_table["Accuracy"].rank(ascending=False),
}).astype(int)
display(rank)
print("Nếu thứ hạng thay đổi giữa các cột (ví dụ hạng 1 ở CV nhưng hạng 3 ở Test) thì đó là dấu hiệu "
      "chênh lệch giữa các mô hình nhỏ hơn mức nhiễu của dữ liệu.")

## 12. Lưu kết quả và dự đoán thử bằng mô hình được chọn

In [ ]:
best_model = final[recommended]
joblib.dump(best_model, "best_model_heart_disease.joblib")
scoreboard.to_csv("model_comparison_summary.csv")
test_table.round(4).to_csv("model_comparison_test.csv")
print("Đã lưu: best_model_heart_disease.joblib, model_comparison_summary.csv, model_comparison_test.csv")
if IN_COLAB:
    for f in ["best_model_heart_disease.joblib", "model_comparison_summary.csv", "model_comparison_test.csv"]:
        files.download(f)

# Bệnh nhân giả lập (dữ liệu thô, chưa qua xử lí) -> dùng lại hàm clean()
demo = pd.DataFrame([
    {"Age": 40, "Sex": "M", "ChestPainType": "ATA", "RestingBP": 130, "Cholesterol": 209, "FastingBS": 0,
     "RestingECG": "Normal", "MaxHR": 165, "ExerciseAngina": "N", "Oldpeak": 0.0, "ST_Slope": "Up"},
    {"Age": 52, "Sex": "M", "ChestPainType": "NAP", "RestingBP": 140, "Cholesterol": 230, "FastingBS": 0,
     "RestingECG": "Normal", "MaxHR": 135, "ExerciseAngina": "N", "Oldpeak": 1.0, "ST_Slope": "Flat"},
    {"Age": 62, "Sex": "M", "ChestPainType": "ASY", "RestingBP": 150, "Cholesterol": 0, "FastingBS": 1,
     "RestingECG": "ST", "MaxHR": 105, "ExerciseAngina": "Y", "Oldpeak": 2.5, "ST_Slope": "Flat"},
])
out = demo[["Age", "Sex", "ChestPainType", "ExerciseAngina", "Oldpeak", "ST_Slope"]].copy()
for n in models:
    out[f"{n} (%)"] = (final[n].predict_proba(clean(demo))[:, 1] * 100).round(1)
out

## 13. Cách đọc kết quả và lưu ý khi viết báo cáo

- **Ưu tiên kết quả Repeated K-Fold** (mục 7, 8) hơn kết quả Test (mục 9): CV dùng nhiều fold nên ổn định hơn nhiều so với một lần đo trên tập Test nhỏ. Khoảng tin cậy ở mục 9.1 thường rộng vài điểm phần trăm, nên chênh lệch dưới mức đó không nên được diễn giải là "mô hình A tốt hơn B".
- **"Không có khác biệt có ý nghĩa" khác với "chứng minh bằng nhau".** Với 918 mẫu, các kiểm định có độ mạnh hạn chế; nên viết "không đủ bằng chứng để phân biệt".
- **Chọn theo nguyên tắc tiết kiệm** (cùng hiệu năng thì chọn mô hình đơn giản/dễ giải thích hơn) là cách biện minh phổ biến, đặc biệt trong y tế, nơi khả năng giải thích có giá trị.
- **Nếu đề tài nhấn mạnh Random Forest,** có thể trình bày: Random Forest là mô hình chính, Logistic Regression và XGBoost là đối chứng; nếu cả ba ngang nhau thì kết luận hiệu năng bị giới hạn bởi dữ liệu (918 mẫu, 11 đặc trưng) hơn là bởi thuật toán. Có thể đổi `PARSIMONY_ORDER` hoặc `PRIMARY_METRIC` để xem quy tắc chọn thay đổi ra sao.
- **Với bài toán y tế,** Recall thường quan trọng hơn Accuracy. Có thể tăng `MIN_RECALL` (ví dụ 0,90) hoặc hạ ngưỡng phân loại dưới 0,5 để giảm bỏ sót ca bệnh.
- **Hạn chế:** dữ liệu nhỏ, thiếu ngẫu nhiên ở Cholesterol (nhóm thiếu có tỉ lệ bệnh rất cao), chưa kiểm chứng trên dữ liệu bên ngoài. Mô hình mang tính tham khảo học thuật, **không thay thế chẩn đoán y khoa**.